# Pipeline ETL — EV3: Requerimientos de Hardware para Juegos de PC

Este notebook documenta y demuestra el pipeline ETL completo que integra tres fuentes de datos:

| # | Fuente | Descripción |
|---|---|---|
| 1 | **Kaggle** — [baraazaid/pc-video-game-requirements](https://www.kaggle.com/datasets/baraazaid/pc-video-game-requirements) | Requisitos mínimos de +80k juegos |
| 2 | **jdegene/steamHWsurvey** | Hardware real de jugadores de Steam (Windows + Linux) |
| 3 | **PCPartPicker** (stub) | Precios en tiempo real — columnas reservadas para la rama `api` |

**Ejecutar el pipeline completo:** `python etl/run_etl.py`

In [ ]:
import pandas as pd
import sys
from pathlib import Path

# Asegurar que el directorio raiz esta en el path
ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
print('Listo.')

---
## 1. Fuente 1: Dataset de Juegos (Kaggle)
### 1.1 Dataset raw

In [ ]:
raw = pd.read_csv('data/kaggle/PC_video_games_requirements.csv')
print(f'Shape: {raw.shape}')
print(f'Columnas: {raw.columns.tolist()}')
raw.head(5)

In [ ]:
print('Nulos por columna:')
print(raw.isnull().sum())
print(f'\nFilas duplicadas: {raw.duplicated().sum()}')

### 1.2 Aplicar limpieza (clean_games.py)
Transformaciones: renombrar columnas, limpiar nombres, normalizar Unknown → NaN, eliminar filas vacías.

In [ ]:
import etl.clean_games as cg
games = cg.run()
print(f'\nDataset limpio: {games.shape}')
games.head(5)

In [ ]:
print('Nulos tras limpieza:')
print(games.isnull().sum())
print(f'\nRAM más comunes:')
print(games['ram'].value_counts().head(8))

---
## 2. Fuente 2: Steam Hardware Survey
### 2.1 Dataset raw

In [ ]:
shs_raw = pd.read_csv('data/steamhwsurvey/shs_platform.csv', parse_dates=['date'])
print(f'Shape: {shs_raw.shape}')
print(f'Plataformas: {shs_raw["platform"].value_counts().to_dict()}')
print(f'Rango de fechas: {shs_raw["date"].min().date()} → {shs_raw["date"].max().date()}')
shs_raw.head(3)

### 2.2 Filtrado (filter_steam_hw.py)
Elimina Mac, conserva Windows (pc) y Linux con categorías de hardware relevantes.

In [ ]:
import etl.filter_steam_hw as fsw
outputs = fsw.run()
steam_latest = outputs['shs_platform_latest']
print(f'\nSnapshot más reciente: {steam_latest.shape}')
print(f'Categorías: {steam_latest["category"].nunique()}')
steam_latest.head(5)

In [ ]:
# Distribución de RAM en jugadores de Windows (mayo 2026)
ram_data = steam_latest[
    (steam_latest['platform'] == 'pc') &
    (steam_latest['category'].str.contains('System RAM', na=False))
][['name', 'percentage']].sort_values('percentage', ascending=False)

print('RAM más común entre jugadores de Windows:')
print(ram_data.to_string(index=False))

In [ ]:
# Top 10 GPUs en Steam (Windows, mayo 2026)
gpu_data = steam_latest[
    (steam_latest['platform'] == 'pc') &
    (steam_latest['category'].str.contains('Video Card Description', na=False))
][['name', 'percentage']].sort_values('percentage', ascending=False).head(10)

print('Top 10 GPUs en Steam (Windows):')
print(gpu_data.to_string(index=False))

---
## 3. Fuente 3: PCPartPicker (stub para API)

Los precios en tiempo real son responsabilidad de la rama `api`.
El ETL reserva estas columnas vacías en el dataset integrado:

| Columna | Descripción |
|---|---|
| `price_cpu_usd` | Precio del CPU recomendado |
| `price_ram_usd` | Precio del kit de RAM recomendado |
| `price_gpu_usd` | Precio de la GPU recomendada |
| `price_storage_usd` | Precio del almacenamiento recomendado |
| `total_upgrade_usd` | Costo total estimado de upgrade |

---
## 4. Integración de las 3 fuentes
### 4.1 Ejecutar integrate.py

In [ ]:
import etl.integrate as integrate
df = integrate.run()
print(f'Dataset integrado: {df.shape}')
df[['game_name','cpu','ram','gpu','ram_market_pct','gpu_market_pct',
    'price_gpu_usd','price_ram_usd','total_upgrade_usd']].head(8)

### 4.2 Cobertura del cruce

In [ ]:
total = len(df)
ram_ok  = df['ram_market_pct'].notna().sum()
gpu_ok  = df['gpu_market_pct'].notna().sum()

print(f'Total de juegos           : {total:,}')
print(f'Con % mercado de RAM      : {ram_ok:,} ({ram_ok/total*100:.1f}%)')
print(f'Con % mercado de GPU      : {gpu_ok:,} ({gpu_ok/total*100:.1f}%)')
print(f'Columnas stub para API    : price_cpu_usd, price_ram_usd, price_gpu_usd, price_storage_usd, total_upgrade_usd')

### 4.3 Ejemplo de uso: ¿Cuántos jugadores pueden correr *Cyberpunk 2077*?

In [ ]:
juego = df[df['game_name'].str.contains('Cyberpunk', na=False, case=False)].iloc[0]

print(f"Juego         : {juego['game_name']}")
print(f"CPU req.      : {juego['cpu']}")
print(f"RAM req.      : {juego['ram']}")
print(f"GPU req.      : {juego['gpu']}")
print(f"Almacenamiento: {juego['storage']}")
print()
if pd.notna(juego['ram_market_pct']):
    print(f"Jugadores con >= RAM requerida : {juego['ram_market_pct']*100:.1f}%")
if pd.notna(juego['gpu_market_pct']):
    print(f"Jugadores con esa GPU exacta   : {juego['gpu_market_pct']*100:.2f}%")
print()
print("Precios (pendiente - rama api):")
print(f"  GPU  : ${juego['price_gpu_usd']}")
print(f"  RAM  : ${juego['price_ram_usd']}")
print(f"  Total: ${juego['total_upgrade_usd']}")

---
## 5. Validación de esquemas
Demostración de la validación con `validate.py`.

In [ ]:
import logging
import etl.validate as validate
from etl.schemas import GAMES_SCHEMA, STEAM_SCHEMA

logger = logging.getLogger('notebook')

_, report_games = validate.validate_dataframe(games, GAMES_SCHEMA, logger)
print('Reporte games_clean:')
print(f"  Filas         : {report_games['rows']:,}")
print(f"  Pasó          : {report_games['passed']}")
print(f"  Advertencias  : {len(report_games['advertencias'])}")

_, report_steam = validate.validate_dataframe(steam_latest, STEAM_SCHEMA, logger)
print('\nReporte shs_platform_latest:')
print(f"  Filas         : {report_steam['rows']:,}")
print(f"  Pasó          : {report_steam['passed']}")
print(f"  Advertencias  : {len(report_steam['advertencias'])}")

---
## 6. Ejecutar el pipeline completo

Para reproducir todos los pasos desde cero:

```bash
python etl/run_etl.py
```

El log completo se guarda en `logs/etl.log`.

**Outputs generados:**
```
data/
├── kaggle/
│   ├── PC_video_games_requirements.csv   ← raw
│   └── games_clean.csv                   ← limpio
├── steamhwsurvey/
│   ├── shs_platform_filtered.csv         ← histórico pc+linux
│   └── shs_platform_latest.csv           ← snapshot mayo 2026
└── integrated/
    └── requirements_market.csv           ← dataset final integrado
```